# 4 - Final statistical analysis

In [ ]:
import digitalhub as dh

In [ ]:
project = dh.get_or_create_project("cflv-1")

## Log inputs

Reports must come from step 3 with `exclude_external_ssl=True` and matching repetition settings and cohort artifact versions.

In [ ]:
artifact_names = {
    "training_results": "2-abmil-conch-v15-results",
    "collection": "3-collection",
    "fm_shift": "3-fm-shift",
    "layer_shift": "3-abmil-layer-shift",
    "top64_shift": "3-abmil-top64-shift",
}

entity_ids = {}  # Optional: {"training_results": "<entity-id>", ...}
inputs = {}
for name, artifact_name in artifact_names.items():
    version = {"entity_id": entity_ids[name]} if name in entity_ids else {}
    inputs[name] = project.get_artifact(artifact_name, **version)
    print(name, inputs[name].key)

## Enrichment definitions

Generator of `training_runs_with_shift_metrics.csv` parameters:
- Entropy: `-sum(p * ln(p))`, with natural logarithms and no normalization.
- Confidence: maximum class probability; margin: largest minus second-largest probability.
- Entropy shift: external mean minus internal mean.
- Confidence/margin drop: internal mean minus external mean.
- External predictions use the original pure-SSL filename exclusion.

## Declaration

In [ ]:
statistics_function = project.new_function(
    name="4-statistical-analysis", kind="python", python_version="PYTHON3_11",
    requirements="4-statistics/requirements.txt", code_src="4-statistics",
    handler="main:analyze_statistics",
)
print(statistics_function.key)

In [ ]:
statistics_function = project.get_function("4-statistical-analysis",
    entity_id="<entity-id>"
)

## Building

In [ ]:
statistics_build = statistics_function.run(action="build",
    resources={"cpu": "8", "mem": "16Gi", "disk": "10Gi"},
)

In [ ]:
statistics_image = statistics_function.refresh().spec.image

## Running

In [ ]:
output_artifact_name = "4-statistics"
statistics_run = statistics_function.run(action="job",
inputs={name: artifact.key for name, artifact in inputs.items()},
    parameters={
        "output_artifact_name": output_artifact_name,
        "include_illustration": False
    },
    resources={"cpu": "8", "mem": "16Gi", "disk": "25Gi"},
    local_execution=False,
)
print(statistics_run.key)